# Analisis Lokasi Potensial Toko Ritel di Kota Bengkulu

Studi kasus berdasarkan materi *Karier dan Kompetisi di Era Industri dengan GIS*: pada bisnis ritel, GIS membantu menentukan lokasi toko berdasarkan jumlah penduduk dan jarak antarcabang.

**Pertanyaan:** area mana di Kota Bengkulu yang padat penduduk, masih jauh dari minimarket/supermarket yang ada, dan dekat jalan utama?

**Pendekatan (multi-criteria, berbasis grid 500 m):**
1. Populasi per sel grid (WorldPop via Google Earth Engine).
2. Jarak ke toko yang sudah ada (OpenStreetMap).
3. Jarak ke jalan utama (OpenStreetMap).
4. Skor gabungan berbobot, lalu uji sensitivitas bobot.

Ini analisis **awal berbasis data terbuka**. Bobot dan ambang adalah asumsi, bukan keputusan bisnis.

## 1. Setup

In [ ]:
!pip -q install osmnx folium

import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm
import osmnx as ox
import ee
from shapely.geometry import box

try:
    ee.Initialize(project="true-campus-479905-q0")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="true-campus-479905-q0")

CRS_M = "EPSG:32748"   # UTM 48S, satuan meter, cocok untuk Bengkulu
print("Siap. osmnx", ox.__version__)

## 2. Batas wilayah studi (Kota Bengkulu)

In [ ]:
def ambil_batas():
    try:
        g = ox.geocode_to_gdf("Kota Bengkulu, Bengkulu, Indonesia")
        if g.geometry.iloc[0].geom_type in ("Polygon", "MultiPolygon"):
            return g[["geometry"]].to_crs(4326)
    except Exception as e:
        print("Geocoding gagal:", e)
    print("Memakai bounding box perkiraan sebagai cadangan.")
    return gpd.GeoDataFrame(geometry=[box(102.20, -3.90, 102.37, -3.70)], crs=4326)

wilayah = ambil_batas()
poly = wilayah.union_all() if hasattr(wilayah, "union_all") else wilayah.unary_union
wilayah_m = wilayah.to_crs(CRS_M)
poly_m = wilayah_m.union_all() if hasattr(wilayah_m, "union_all") else wilayah_m.unary_union
print(f"Luas wilayah studi: {poly_m.area/1e6:.1f} km2")

## 3. Data toko dan jalan dari OpenStreetMap

In [ ]:
tags_toko = {"shop": ["convenience", "supermarket"]}
toko = ox.features_from_polygon(poly, tags_toko).reset_index(drop=True)[["geometry"]]
toko = toko.to_crs(CRS_M)
toko["geometry"] = toko.geometry.centroid
print("Jumlah toko (convenience + supermarket) di OSM:", len(toko))
assert len(toko) > 0, "Tidak ada toko ditemukan di OSM untuk wilayah ini."

tags_jalan = {"highway": ["primary", "secondary", "tertiary"]}
jalan = ox.features_from_polygon(poly, tags_jalan).reset_index(drop=True)[["geometry"]]
jalan = jalan[jalan.geometry.geom_type.isin(["LineString", "MultiLineString"])].to_crs(CRS_M)
print("Jumlah segmen jalan utama:", len(jalan))

> Catatan: OSM dibuat sukarelawan, jadi toko yang belum dipetakan tidak akan terhitung. Hasil analisis sebaiknya diverifikasi di lapangan.

## 4. Grid analisis 500 m

In [ ]:
SEL = 500  # meter
minx, miny, maxx, maxy = poly_m.bounds
cells = [box(x, y, x + SEL, y + SEL)
         for x in np.arange(minx, maxx, SEL)
         for y in np.arange(miny, maxy, SEL)]
grid = gpd.GeoDataFrame({"id": range(len(cells))}, geometry=cells, crs=CRS_M)
grid = grid[grid.centroid.within(poly_m)].reset_index(drop=True)
grid["id"] = range(len(grid))
print("Jumlah sel grid:", len(grid))

## 5. Populasi per sel (WorldPop via Earth Engine)

In [ ]:
koleksi = (ee.ImageCollection("WorldPop/GP/100m/pop")
           .filter(ee.Filter.eq("country", "IDN"))
           .filter(ee.Filter.eq("year", 2020)))
assert koleksi.size().getInfo() > 0, "Koleksi WorldPop kosong; cek nama properti/dataset."
pop_img = koleksi.select("population").mosaic()

fc = ee.FeatureCollection(json.loads(grid[["id", "geometry"]].to_crs(4326).to_json()))
hasil = pop_img.reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=100).getInfo()

pop = {f["properties"]["id"]: f["properties"].get("sum", 0) or 0 for f in hasil["features"]}
grid["populasi"] = grid["id"].map(pop).astype(float)
print(f"Total populasi terestimasi: {grid.populasi.sum():,.0f} jiwa")
grid["populasi"].describe().round(1)

## 6. Jarak ke toko yang ada dan ke jalan utama

In [ ]:
cent = grid[["id"]].copy()
cent["geometry"] = grid.centroid
cent = gpd.GeoDataFrame(cent, crs=CRS_M)

j_toko = gpd.sjoin_nearest(cent, toko[["geometry"]], distance_col="jarak_toko_m")
grid["jarak_toko_m"] = grid["id"].map(j_toko.groupby("id")["jarak_toko_m"].min())

j_jalan = gpd.sjoin_nearest(cent, jalan[["geometry"]], distance_col="jarak_jalan_m")
grid["jarak_jalan_m"] = grid["id"].map(j_jalan.groupby("id")["jarak_jalan_m"].min())

grid[["populasi", "jarak_toko_m", "jarak_jalan_m"]].describe().round(1)

## 7. Skor kelayakan

| Kriteria | Aturan | Bobot awal |
| --- | --- | --- |
| Populasi | makin padat makin baik (min-max) | 0,5 |
| Celah layanan | makin jauh dari toko ada makin baik, dibatasi 1000 m | 0,3 |
| Akses jalan | makin dekat jalan utama makin baik, dibatasi 1000 m | 0,2 |

Sel dengan populasi di bawah ambang (200 jiwa) dianggap tidak layak. Semua angka ini **asumsi** dan bisa diubah.

In [ ]:
BOBOT = {"pop": 0.5, "celah": 0.3, "jalan": 0.2}
MIN_POP = 200
CAP_TOKO = 1000
CAP_JALAN = 1000

def minmax(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0

grid["s_pop"] = minmax(grid["populasi"])
grid["s_celah"] = grid["jarak_toko_m"].clip(upper=CAP_TOKO) / CAP_TOKO
grid["s_jalan"] = 1 - grid["jarak_jalan_m"].clip(upper=CAP_JALAN) / CAP_JALAN
grid["layak"] = grid["populasi"] >= MIN_POP

def hitung_skor(b):
    s = b["pop"] * grid["s_pop"] + b["celah"] * grid["s_celah"] + b["jalan"] * grid["s_jalan"]
    return s.where(grid["layak"])

grid["skor"] = hitung_skor(BOBOT)
print("Sel layak:", int(grid["layak"].sum()), "dari", len(grid))

kandidat = grid[grid["layak"]].nlargest(10, "skor").copy()
kandidat["peringkat"] = range(1, len(kandidat) + 1)
kandidat[["peringkat", "id", "populasi", "jarak_toko_m", "jarak_jalan_m", "skor"]].round(2)

## 8. Uji sensitivitas bobot

In [ ]:
skenario = {
    "dasar":          {"pop": 0.5, "celah": 0.3, "jalan": 0.2},
    "fokus populasi": {"pop": 0.7, "celah": 0.2, "jalan": 0.1},
    "fokus celah":    {"pop": 0.3, "celah": 0.5, "jalan": 0.2},
    "fokus akses":    {"pop": 0.3, "celah": 0.2, "jalan": 0.5},
    "merata":         {"pop": 1/3, "celah": 1/3, "jalan": 1/3},
}
top = {}
for nama, b in skenario.items():
    s = hitung_skor(b)
    top[nama] = set(grid.loc[s.nlargest(10).index, "id"])

dasar = top["dasar"]
for nama, ids in top.items():
    print(f"{nama:15s} irisan dengan skenario dasar: {len(ids & dasar)}/10")

stabil = set.intersection(*top.values())
print("\nSel yang selalu masuk 10 besar di semua skenario:", sorted(stabil))

Sel yang bertahan di semua skenario adalah kandidat yang paling kokoh terhadap perubahan asumsi bobot.

## 9. Peta statis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
grid.plot(column="skor", cmap="YlOrRd", legend=True, ax=ax,
          missing_kwds={"color": "lightgrey"}, edgecolor="none",
          legend_kwds={"label": "Skor kelayakan"})
toko.plot(ax=ax, color="blue", markersize=8, label="Toko ada (OSM)")
kandidat.assign(geometry=kandidat.centroid).plot(ax=ax, color="black", marker="*",
                                                  markersize=120, label="10 kandidat teratas")
jalan.plot(ax=ax, color="grey", linewidth=0.5)
ax.set_title("Skor kelayakan lokasi toko, Kota Bengkulu (grid 500 m)")
ax.legend(loc="lower left"); ax.set_axis_off()
plt.tight_layout(); plt.savefig("peta_skor_lokasi_toko.png", dpi=150); plt.show()

## 10. Peta interaktif (web map)

In [ ]:
g4326 = grid.to_crs(4326)
skor_valid = g4326["skor"].dropna()
cmap = cm.LinearColormap(["#ffffb2", "#fd8d3c", "#bd0026"],
                         vmin=float(skor_valid.min()), vmax=float(skor_valid.max()),
                         caption="Skor kelayakan")

pusat = [g4326.geometry.centroid.y.mean(), g4326.geometry.centroid.x.mean()]
m = folium.Map(location=pusat, zoom_start=12, tiles="CartoDB positron")

folium.GeoJson(
    g4326[["id", "populasi", "jarak_toko_m", "jarak_jalan_m", "skor", "geometry"]].round(2),
    style_function=lambda f: {
        "fillColor": cmap(f["properties"]["skor"]) if f["properties"]["skor"] is not None else "#cccccc",
        "color": "none", "fillOpacity": 0.6},
    tooltip=folium.GeoJsonTooltip(fields=["id", "populasi", "jarak_toko_m", "jarak_jalan_m", "skor"]),
    name="Skor grid"
).add_to(m)

for _, r in toko.to_crs(4326).iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=3, color="blue", fill=True).add_to(m)

for _, r in kandidat.to_crs(4326).iterrows():
    c = r.geometry.centroid
    folium.Marker([c.y, c.x], tooltip=f"Peringkat {int(r['peringkat'])} (skor {r['skor']:.2f})",
                  icon=folium.Icon(color="black", icon="star")).add_to(m)

cmap.add_to(m)
m.save("peta_lokasi_toko_bengkulu.html")
m

## 11. Simpan hasil

In [ ]:
grid.drop(columns="geometry").round(3).to_csv("hasil_skor_grid_bengkulu.csv", index=False)
kandidat.to_crs(4326)[["peringkat", "id", "populasi", "jarak_toko_m", "jarak_jalan_m", "skor", "geometry"]] \
    .to_file("kandidat_lokasi_toko_bengkulu.geojson", driver="GeoJSON")
print("Tersimpan: hasil_skor_grid_bengkulu.csv, kandidat_lokasi_toko_bengkulu.geojson,",
      "peta_skor_lokasi_toko.png, peta_lokasi_toko_bengkulu.html")

## 12. Interpretasi dan keterbatasan

- Populasi WorldPop adalah **estimasi berbasis model**, bukan sensus per sel.
- Toko dan jalan dari OSM bisa belum lengkap.
- Jarak memakai garis lurus, bukan waktu tempuh jaringan jalan.
- Bobot, batas 1000 m, dan ambang 200 jiwa adalah asumsi. Uji sensitivitas menunjukkan seberapa kokoh hasilnya, tetapi belum memperhitungkan harga lahan, pesaing non-OSM, dan daya beli.
- Hasil adalah **daftar area untuk survei lanjutan**, bukan rekomendasi final lokasi.

**Kaitan dengan materi:** studi ini memakai Python (GeoPandas), API data terbuka (OSM dan Earth Engine), analisis jarak, visualisasi peta web, dan keluaran GeoJSON yang bisa dipakai di GIS lain. Pesan materi tentang portofolio dan penyelesaian masalah nyata tercermin pada alur: pertanyaan, data, analisis, uji asumsi, dan batasan.